In [4]:
import json
import time
import paho.mqtt.client as mqtt
import logging

In [7]:
def sender(data_list):
    logging.basicConfig(level=logging.INFO)
    logger = logging.getLogger(__name__)

    # Настройки MQTT
    # BROKER = "158.160.240.169"
    BROKER="93.183.71.105"
    PORT = 1883
    # PORT = 8084
    DEVICE_SERIAL = "00000021"
    USERNAME = DEVICE_SERIAL
    PASSWORD = "FISI00000021"

    MQTT_TOPIC_DATA = f"mqtt/devices/{DEVICE_SERIAL}/data/"

    # Callback
    def on_connect(client, userdata, flags, reason_code, properties):
        if reason_code == 0:
            logger.info("Connected to MQTT Broker")
            pass
        else:
            logger.error(f"Failed to connect, reason_code={reason_code}")

    def on_publish(client, userdata, mid, reason_code, properties):
        logger.info(f"Message {mid} published with reason code {reason_code}")

    # Создаём клиент и подключаемся
    client = mqtt.Client(
        client_id=DEVICE_SERIAL, callback_api_version=mqtt.CallbackAPIVersion.VERSION2, 
    )
    # if USERNAME:
    client.username_pw_set(USERNAME, PASSWORD)

    client.on_connect = on_connect
    client.on_publish = on_publish

    client.connect(BROKER, PORT)
    client.loop_start()

    # Публикуем каждую запись по очереди

    for entry in data_list:
        payload = json.dumps(entry)
        client.publish(MQTT_TOPIC_DATA, payload=payload, qos=1)
        logger.info(f"Published: {payload}")
        time.sleep(0.1)  # пауза между публикациями

    client.loop_stop()
    client.disconnect()
    logger.info("Disconnected from MQTT")

In [8]:
sender(['aaaa','bbbb'])

INFO:__main__:Published: "aaaa"
INFO:__main__:Connected to MQTT Broker
INFO:__main__:Message 1 published with reason code Success
INFO:__main__:Published: "bbbb"
INFO:__main__:Message 2 published with reason code Success
INFO:__main__:Disconnected from MQTT
